In [ ]:
import os
import anndata as ad
import numpy as np
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import omicverse as ov
import scvi
from scvi.model.utils import mde
from scarches.models.scpoli import scPoli
from scarches.dataset.trvae.data_handling import remove_sparsity

import warnings
warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2

In [2]:
sc.settings.set_figure_params(dpi=100, frameon=False)
sc.set_figure_params(dpi=100)
sc.set_figure_params(figsize=(3, 3))
plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.figsize'] = (3, 3)

In [ ]:
# Change the working directory to the Garfield folder (if needed)
os.chdir('/storage2/liuxiaodongLab/fanxueying/embryo_benchmarking_rebuttal/code/20250729_scpoli_optimization_v3')
os.getcwd()

In [4]:
embryomodel = ad.read_h5ad("/storage2/liuxiaodongLab/fanxueying/embryo_benchmarking_rebuttal/code/20250729_scpoli_optimization_v3/embryo_model_integration_scPoli.h5ad")

In [ ]:
embryomodel

In [ ]:
sc.pl.umap(
    embryomodel,
    color='human_ref_reanno_pred',
    show=False,
    frameon=False,
)

In [ ]:
sc.pl.umap(
    embryomodel,
    color='orig.ident',
    show=False,
    frameon=False,
)

In [ ]:
sc.pl.umap(
    embryomodel,
    color='scANVI_res_0.5',
    show=False,
    frameon=False,
)

In [ ]:
import scanpy as sc
import pandas as pd

# Ensure you have the AnnData object 'embryomodel' loaded
# Example: embryomodel = sc.read_h5ad("path/to/your/file.h5ad")

# Extract the metadata (obs) we need
df = embryomodel.obs[['orig.ident', 'human_ref_reanno_pred']].copy()

# Optional: Check for missing values
print("Missing human_ref_reanno_pred entries:", df['human_ref_reanno_pred'].isnull().sum())

# Drop rows with NaN if needed (uncomment below if necessary)
# df.dropna(subset=['human_ref_reanno_pred'], inplace=True)

# Create contingency table: orig.ident x human_ref_reanno_pred
summary_table = pd.crosstab(
    df['orig.ident'],
    df['human_ref_reanno_pred']
)

# Display the result
print(summary_table)

# Optional: Save to CSV
summary_table.to_csv("./human_ref_reanno_pred_by_orig_ident.csv")

# Optional: Also save normalized version (row-wise %, i.e., proportions within each sample)
proportion_table = summary_table.divide(summary_table.sum(axis=1), axis=0)
proportion_table.to_csv("./human_ref_reanno_pred_proportions_by_orig_ident.csv")

# Bonus: Show top predicted cell types per sample
top_cell_types_per_sample = summary_table.idxmax(axis=1).to_dict()
print("\nTop 'human_ref_reanno_pred' per sample:")
for sample, cell_type in top_cell_types_per_sample.items():
    print(f"  {sample}: {cell_type}")

In [11]:
# Step 1: Identify indices of high-confidence cells
# Step 1: Identify indices of high-confidence cells
high_confidence_indices = np.where(
    ((embryomodel.obs["human_ref_lineage_uncert"] < 0.1) &  
     (embryomodel.obs["human_ref_lineage_pred"] != "PGC")) |
    ((embryomodel.obs["scANVI_res_0.5"] == '6') & 
     (embryomodel.obs["human_ref_lineage_pred"] == "PGC") )
)[0]

# Step 2: Extract high-confidence cells
high_confidence_cells = embryomodel[high_confidence_indices]

# Step 3: Extract high-confidence labels (cell type annotations)
high_confidence_labels = embryomodel.obs["human_ref_lineage_pred"].iloc[high_confidence_indices]


In [ ]:
# Ensure UMAP coordinates exist
if "X_umap" not in embryomodel.obsm:
    raise ValueError("UMAP coordinates (embryomodel.obsm['X_umap']) are missing. Please compute UMAP first.")

# Extract UMAP coordinates for all cells
umap_coords = embryomodel.obsm["X_umap"]

# Create a mask for high-confidence cells
is_high_confidence = np.zeros(embryomodel.shape[0], dtype=bool)  # Initialize mask
is_high_confidence[high_confidence_indices] = True         # Mark high-confidence cells

# Plot all cells in grey
plt.figure(figsize=(8, 6))
# Plot all cells in grey
plt.scatter(
    umap_coords[~is_high_confidence, 0],
    umap_coords[~is_high_confidence, 1],
    c="grey",
    s=10,
    label="Other Cells"
)

# Overlay high-confidence cells colored by cell type
for label in set(high_confidence_labels):
    label_indices = high_confidence_indices[high_confidence_labels == label]
    plt.scatter(
        umap_coords[label_indices, 0],
        umap_coords[label_indices, 1],
        label=label,
        s=50
    )

# Add legend and finalize plot
plt.title("UMAP with High-Confidence Cells Colored by Cell Type")
plt.xlabel("UMAP-1")
plt.ylabel("UMAP-2")
# Move legend outside the frame
plt.legend(
    loc="upper left",                     # Position relative to the plot
    bbox_to_anchor=(1.02, 1),             # Offset to place legend outside
    borderaxespad=0,                      # Padding between legend and axes
    fontsize=10                           # Optional: Adjust font size
)

# Adjust layout to prevent clipping
plt.tight_layout()

# Show the plot
plt.show()

In [ ]:
# Step 3: Extract high-confidence labels (cell type annotations)
high_confidence_labels = embryomodel.obs["human_ref_reanno_pred"][high_confidence_indices]

# Ensure UMAP coordinates exist
if "X_umap" not in embryomodel.obsm:
    raise ValueError("UMAP coordinates (embryomodel.obsm['X_umap']) are missing. Please compute UMAP first.")

# Extract UMAP coordinates for all cells
umap_coords = embryomodel.obsm["X_umap"]

# Create a mask for high-confidence cells
is_high_confidence = np.zeros(embryomodel.shape[0], dtype=bool)  # Initialize mask
is_high_confidence[high_confidence_indices] = True         # Mark high-confidence cells

# Plot all cells in grey
plt.figure(figsize=(8, 6))
# Plot all cells in grey
plt.scatter(
    umap_coords[~is_high_confidence, 0],
    umap_coords[~is_high_confidence, 1],
    c="grey",
    s=10,
    label="Other Cells"
)

# Overlay high-confidence cells colored by cell type
for label in set(high_confidence_labels):
    label_indices = high_confidence_indices[high_confidence_labels == label]
    plt.scatter(
        umap_coords[label_indices, 0],
        umap_coords[label_indices, 1],
        label=label,
        s=30
    )

# Add legend and finalize plot
plt.title("UMAP with High-Confidence Cells Colored by Cell Type")
plt.xlabel("UMAP-1")
plt.ylabel("UMAP-2")
# Move legend outside the frame
plt.legend(
    loc="upper left",                     # Position relative to the plot
    bbox_to_anchor=(1.02, 1),             # Offset to place legend outside
    borderaxespad=0,                      # Padding between legend and axes
    fontsize=5                           # Optional: Adjust font size
)

# Adjust layout to prevent clipping
plt.tight_layout()

# Show the plot
plt.show()

In [ ]:
# 1. Subset adata, only keep high_confidence_indices
query_adata = high_confidence_cells

# 2. For the subset adata, if the human_ref_lineage_pred of the remaining cells is PGC, modify the cell's human_ref_reanno_pred to PGC
pgc_mask = query_adata.obs["human_ref_lineage_pred"] == "PGC"

# Check if "PGC" is already in the categories of human_ref_reanno_pred
if "PGC" not in query_adata.obs["human_ref_reanno_pred"].cat.categories:
    # If not, first add "PGC" to the categories
    query_adata.obs["human_ref_reanno_pred"] = query_adata.obs["human_ref_reanno_pred"].cat.add_categories(["PGC"])
    
pgc_mask = query_adata.obs["human_ref_lineage_pred"] == "PGC"
query_adata.obs.loc[pgc_mask, "human_ref_reanno_pred"] = "PGC"

# Verify the modification results
print(f"Total number of cells: {len(query_adata)}")
print(f"Number of PGC cells: {sum(query_adata.obs['human_ref_lineage_pred'] == 'PGC')}")
print(f"Number of PGC cells in human_ref_reanno_pred: {sum(query_adata.obs['human_ref_reanno_pred'] == 'PGC')}")

In [15]:
# exclude_lineages = ["Primitive.streak", "Amniotic_ecto", "NMP"]
exclude_lineages = ["Primitive.streak", "Amniotic_ecto", "NMP", "Endoderm", "ExE_endo"]
mask = ~query_adata.obs["human_ref_lineage_pred"].isin(exclude_lineages)
query_adata = query_adata[mask].copy()

In [ ]:
query_adata

In [ ]:
print(query_adata.layers["counts"])

In [18]:
counts_matrix = query_adata.layers["counts"].toarray()
query_adata = sc.AnnData(
            X=counts_matrix,
            obs=query_adata.obs.copy(),
            var=query_adata.var.copy(),
            layers={'counts': counts_matrix}
        )
query_adata = remove_sparsity(query_adata)

In [ ]:
query_adata

In [ ]:
#load reference lineage model 
source_adata = sc.read('/storage2/liuxiaodongLab/fanxueying/embryo_benchmarking_rebuttal/code/20250729_scpoli_optimization_v3/lineage_model_hvg2000_dim50_reseed/adata.h5ad')
source_adata


In [ ]:
#counts_matrix = query_adata.layers["counts"].toarray()
query_adata.layers["counts"]

In [22]:
query_adata = sc.AnnData(
    X=counts_matrix,
    obs=query_adata.obs.copy(),
    var=query_adata.var.copy(),
    layers={'counts': counts_matrix}
)
query_adata = remove_sparsity(query_adata)

In [ ]:
query_adata

In [24]:
# Reorganize query dataset to match genes in the reference dataset
all_genes = source_adata.var_names
missing_genes = all_genes.difference(query_adata.var_names)
missing_data = np.zeros((query_adata.shape[0], len(missing_genes)))
query_adata_df = pd.DataFrame(query_adata.X, columns=query_adata.var_names, index=query_adata.obs_names)
missing_df = pd.DataFrame(missing_data, columns=missing_genes, index=query_adata.obs_names)
query_adata_combined_df = pd.concat([query_adata_df, missing_df], axis=1)[all_genes]
query_adata_extended = sc.AnnData(
    X=query_adata_combined_df.values,
    obs=query_adata.obs,
    var=pd.DataFrame(index=all_genes),
    layers={'counts': query_adata_combined_df.values}
)

In [ ]:
query_adata_extended

In [ ]:
# Minimal version - Keep only the most basic information
obs_to_keep = ['orig.ident', 'nCount_RNA', 'nFeature_RNA', 'stage', 'percent.mt', 
               'platform', 'species', 'embryo', 'human_ref_lineage_pred', 
               'human_ref_reanno_pred']
layers_to_keep = ['counts', 'logcounts']

# Check which columns and layers actually exist
existing_obs = [col for col in obs_to_keep if col in query_adata_extended.obs.columns]
existing_layers = [layer for layer in layers_to_keep if layer in query_adata_extended.layers.keys()]

# Create a simplified AnnData object
query_adata_clean = sc.AnnData(
    X=query_adata_extended.X.copy(),
    obs=query_adata_extended.obs[existing_obs].copy(),
    var=pd.DataFrame(index=query_adata_extended.var.index),  # Keep only the gene index
    layers={layer: query_adata_extended.layers[layer].copy() for layer in existing_layers}
)

# Replace the original object
query_adata_extended = query_adata_clean

print("Cleaned adata:")
print(query_adata_extended)

In [ ]:
# reanno
query_adata_extended.obs.rename(columns={
    'human_ref_lineage_pred': 'lineage',
    'human_ref_reanno_pred': 'reanno'
}, inplace=True)

print(f"Updated obs columns: {list(query_adata_extended.obs.columns)}")

In [ ]:
query_adata_extended

In [ ]:
source_adata

In [ ]:
# Concatenate the datasets
combined_adata = sc.concat([source_adata, query_adata_extended], axis=0, join="outer")
combined_adata

In [ ]:
print(combined_adata.obs['lineage'])

In [ ]:
# Retrain the scPoli model with the combined dataset
condition_key = 'orig.ident'
cell_type_key = "lineage"
early_stopping_kwargs = {
    "early_stopping_metric": "val_prototype_loss",
    "mode": "min",
    "threshold": 0,
    "patience": 20,
    "reduce_lr": True,
    "lr_patience": 13,
    "lr_factor": 0.1,
}

print("Retraining scPoli model with enhanced dataset...")
enhanced_scpoli_model = scPoli(
    adata=combined_adata,
    condition_keys=condition_key,
    cell_type_keys=cell_type_key,
    embedding_dims=5,
    recon_loss='nb',
)
enhanced_scpoli_model.train(
    n_epochs=50,
    pretraining_epochs=40,
    early_stopping_kwargs=early_stopping_kwargs,
    eta=5,
)

In [ ]:
enhanced_model_dir='./enhanced_reference_model_lineage/'
os.makedirs(enhanced_model_dir, exist_ok=True)  
# Create directory for saving the model
# Save the enhanced model and AnnData object
try:
    print("Saving the enhanced model...")
    enhanced_scpoli_model.save(enhanced_model_dir, overwrite=True, save_anndata=True)
    print(f"Enhanced model saved to {enhanced_model_dir}")

except Exception as e:
    print(f"Error while saving the enhanced model or AnnData: {e}")

In [34]:
#get latent representation of reference data
enhanced_scpoli_model.model.eval()
data_latent_source = enhanced_scpoli_model.get_latent(
    combined_adata,
    mean=True
)

In [ ]:
# get latent
combined_adata.obsm["X_scpoli"] = data_latent_source

# calculate UMAP
sc.pp.neighbors(combined_adata, use_rep="X_scpoli")
sc.tl.umap(combined_adata)

In [ ]:
# 1. plot by cell type key
plt.figure()
sc.pl.umap(combined_adata, color=cell_type_key, title=f'UMAP by {cell_type_key}', frameon=False, show=True, save="enhanced_lineage_training_label.pdf")

In [ ]:
# 2. plot by batch effect
plt.figure()
sc.pl.umap(combined_adata, color=condition_key, title=f'UMAP by {condition_key} (Batch)', frameon=False, show=True,save="enhanced_lineage_training_batch.pdf")

In [ ]:
Weatherbee = sc.read_h5ad('/storage2/liuxiaodongLab/fanxueying/mayanalysis/2024Aug/garfield/in_vitro_embryo_models/processed/data/corrected_processed_zheng_2022.h5ad')
Weatherbee

In [39]:
sc.settings.seed = 42
Weatherbee.layers["counts"] =Weatherbee.X.copy()
sc.pp.normalize_total(Weatherbee, target_sum=1e4)
sc.pp.log1p(Weatherbee)
Weatherbee.layers["logcounts"] = Weatherbee.X.copy()
sc.pp.highly_variable_genes(Weatherbee, n_top_genes=2000, flavor="cell_ranger", batch_key="orig.ident")
sc.tl.pca(Weatherbee, n_comps=30, use_highly_variable=True)

In [ ]:
Weatherbee

In [41]:
counts_matrix = Weatherbee.layers["counts"].toarray()
adata3 = sc.AnnData(X=counts_matrix, obs=Weatherbee.obs.copy(), var=Weatherbee.var.copy(), layers={'counts': counts_matrix})

In [42]:
adata3 = remove_sparsity(adata3)

In [43]:
all_genes = combined_adata.var_names
missing_genes = all_genes.difference(adata3.var_names)
missing_data = np.zeros((adata3.shape[0], len(missing_genes)))
adata3_df = pd.DataFrame(adata3.X, columns=adata3.var_names, index=adata3.obs_names)
missing_df = pd.DataFrame(missing_data, columns=missing_genes, index=adata3.obs_names)
adata3_combined_df = pd.concat([adata3_df, missing_df], axis=1)
adata3_combined_df = adata3_combined_df[all_genes]
adata3_extended = sc.AnnData(
    X=adata3_combined_df.values, 
    obs=adata3.obs,
    var=pd.DataFrame(index=all_genes),
    layers={'counts': adata3_combined_df.values})
adata3_extended.var['features'] = Weatherbee.var.reindex(all_genes)['features']

In [44]:
#obs_to_keep = ['orig.ident', 'nCount_RNA', 'nFeature_RNA', 'percent.mt', 'sample_type', 'scmap_nakamura', 'scmapCELL_Yang', 'scmap_ma', 'scmap_Tyser', 'scmapCELL_Mole', 'cell_assignment', 'course_cell_assignment', 'stage', 'species', 'embryo', 'platform']
obs_to_keep = ['orig.ident', 'nCount_RNA', 'nFeature_RNA', 'percent.mt', 'species', 'embryo', 'platform', 'doublet', 'doublet_score']
obs_columns_to_remove = [col for col in adata3_extended.obs.columns if col not in obs_to_keep]
adata3_extended.obs.drop(columns=obs_columns_to_remove, inplace=True)

In [45]:
target_adata = adata3_extended.copy()

In [46]:
# Add 'final_anno' column with 'Unknown', otherwise the next line give errors
target_adata.obs['lineage'] = 'Unknown'

In [ ]:
scpoli_query = scPoli.load_query_data(
    adata=target_adata,
    reference_model=enhanced_scpoli_model,
    labeled_indices=[],
)

In [ ]:
scpoli_query.train(
    n_epochs=50,
    pretraining_epochs=40,
    eta=5
)

In [ ]:
print(target_adata.X.dtype)  # This is likely the x tensor
target_adata.X = target_adata.X.astype(np.float32)
print(target_adata.obs.dtypes)  # Check types in the annotations/metadata
print(target_adata.var.dtypes)  # Check types in the variable features (genes)

In [50]:
#Label transfer from reference to query
results_dict = scpoli_query.classify(target_adata, scale_uncertainties=True)

In [ ]:
from sklearn.metrics import classification_report
# check the label transfer performance achieved
for i in range(len(cell_type_key)):
    preds = results_dict[cell_type_key]["preds"]
    results_dict[cell_type_key]["uncert"]
    classification_df = pd.DataFrame(
        classification_report(
            y_true=target_adata.obs[cell_type_key],
            y_pred=preds,
            output_dict=True,
        )
    ).transpose()
print(classification_df)

In [52]:
#get latent representation of reference data
scpoli_query.model.eval()
data_latent_source = scpoli_query.get_latent(
    combined_adata,
    mean=True
)

adata_latent_source = sc.AnnData(data_latent_source)
adata_latent_source.obs = combined_adata.obs.copy()

#get latent representation of query data
data_latent= scpoli_query.get_latent(
    target_adata,
    mean=True
)

adata_latent = sc.AnnData(data_latent)
adata_latent.obs = target_adata.obs.copy()

#get label annotations
adata_latent.obs['lineage_pred'] = results_dict['lineage']['preds'].tolist()
adata_latent.obs['lineage_uncert'] = results_dict['lineage']['uncert'].tolist()
adata_latent.obs['classifier_outcome'] = (
    adata_latent.obs['lineage_pred'] == adata_latent.obs['lineage']
)

#get prototypes
labeled_prototypes = scpoli_query.get_prototypes_info()
labeled_prototypes.obs['study'] = 'labeled prototype'
unlabeled_prototypes = scpoli_query.get_prototypes_info(prototype_set='unlabeled')
unlabeled_prototypes.obs['study'] = 'unlabeled prototype'

#join adatas
adata_latent_full = adata_latent_source.concatenate(
    [adata_latent, labeled_prototypes, unlabeled_prototypes],
    batch_key='query'
)
adata_latent_full.obs['lineage_pred'][adata_latent_full.obs['query'].isin(['0'])] = np.nan
sc.pp.neighbors(adata_latent_full, n_neighbors=15)
sc.tl.umap(adata_latent_full)

In [53]:
#get adata without prototypes
adata_no_prototypes = adata_latent_full[adata_latent_full.obs['query'].isin(['0', '1'])]

In [ ]:
sc.pl.umap(
    adata_no_prototypes,
    color='lineage_pred',
    show=False,
    frameon=False,
)

In [ ]:
sc.pl.umap(
    adata_no_prototypes,
    color='orig.ident',
    show=False,
    frameon=False,
)

In [ ]:
sc.pl.umap(
    adata_no_prototypes,
    color='lineage_uncert',
    show=False,
    frameon=False,
    cmap='magma',
    vmax=1
)

In [ ]:
adata_latent

In [58]:
sc.pp.neighbors(adata_latent)
sc.tl.leiden(adata_latent)
sc.tl.umap(adata_latent)

In [ ]:
sc.pl.umap(
    adata_latent,
    color='lineage_pred',
    show=False,
    frameon=False,
)